# Báo cáo chất lượng dữ liệu tự động cho Olist

Bạn sắp giao bộ Olist cho cả team dùng (EDA, model, dashboard). Trước đó, viết một báo cáo chất
lượng chạy lại được mỗi lần dữ liệu về: 15 kiểm tra có tên, có mức độ, có mẫu số rõ ràng.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật và gọi hàm để xem. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/09-eda-storytelling/exercises/intermediate/01-olist-quality-report -q`

In [ ]:
import numpy as np
import pandas as pd

# (tên kiểm tra, bảng, severity) — đúng thứ tự này trong báo cáo
CHECKS = [
    ("orders_pk_duplicated", "orders", "error"),
    ("delivered_missing_delivery_date", "orders", "warning"),
    ("delivered_missing_approved_at", "orders", "warning"),
    ("carrier_after_customer_delivery", "orders", "error"),
    ("approved_before_purchase", "orders", "error"),
    ("orders_without_items", "orders", "warning"),
    ("items_orphan_order", "items", "error"),
    ("items_orphan_product", "items", "error"),
    ("orders_without_review", "orders", "warning"),
    ("review_id_duplicated", "reviews", "warning"),
    ("products_missing_category", "products", "warning"),
    ("category_not_translated", "products", "warning"),
    ("customer_zip_not_in_geolocation", "customers", "warning"),
    ("payment_items_mismatch", "orders", "warning"),
    ("freight_zero", "items", "info"),
]
REPORT_COLUMNS = ["check", "table", "severity", "n_failed", "n_total", "fail_rate"]

In [ ]:
# explore: đọc 8 bảng Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
order_dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"]
tables = {
    "orders": pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=order_dates),
    "items": pd.read_csv(RAW / "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(RAW / "olist_order_payments_dataset.csv"),
    "reviews": pd.read_csv(RAW / "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(RAW / "olist_products_dataset.csv"),
    "customers": pd.read_csv(RAW / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str}),
    "geolocation": pd.read_csv(RAW / "olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str}),
    "translation": pd.read_csv(RAW / "product_category_name_translation.csv"),
}
{name: len(df) for name, df in tables.items()}

## 1. Ba khối xây dựng

Mọi kiểm tra trong báo cáo đều là một trong ba dạng: khoá lặp, khoá ngoại mồ côi, thứ tự thời gian sai.

In [ ]:
def count_duplicated(df: pd.DataFrame, key: list[str]) -> int:
    """Số dòng có key xuất hiện hơn một lần (đếm mọi bản, không chỉ bản thừa)."""
    # TODO
    raise NotImplementedError


def count_orphans(child: pd.DataFrame, child_key: str, parent: pd.DataFrame, parent_key: str) -> int:
    """Số dòng của child có child_key khác null nhưng không có trong parent[parent_key]."""
    # TODO
    raise NotImplementedError


def count_order_violations(df: pd.DataFrame, earlier: str, later: str) -> tuple[int, int]:
    """(số dòng vi phạm, số dòng được kiểm). Chỉ kiểm dòng có cả hai cột khác null;
    vi phạm khi df[earlier] > df[later]."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thử từng hàm trên dữ liệu thật
print(count_duplicated(tables["reviews"], ["review_id"]))
print(count_orphans(tables["orders"], "order_id", tables["items"], "order_id"))
print(count_order_violations(tables["orders"], "order_delivered_carrier_date", "order_delivered_customer_date"))

## 2. Báo cáo đủ 15 kiểm tra

Bảng định nghĩa `n_failed` / `n_total` từng kiểm tra nằm trong README của bài.

In [ ]:
def quality_report(t: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Chạy đủ 15 kiểm tra trong CHECKS theo đúng thứ tự; trả DataFrame cột REPORT_COLUMNS.

    t có các khoá: orders, items, payments, reviews, products, customers, geolocation, translation.
    Cột ngày của orders đã là datetime. n_failed / n_total của từng kiểm tra: bảng trong README.
    fail_rate = n_failed / n_total (0.0 nếu n_total = 0).
    """
    # TODO: tính (n_failed, n_total) cho từng kiểm tra bằng ba hàm ở trên, rồi dựng DataFrame
    raise NotImplementedError


def blocking_checks(report: pd.DataFrame) -> list[str]:
    """Tên các kiểm tra severity "error" có n_failed > 0, giữ thứ tự trong report."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: báo cáo trên Olist thật
report = quality_report(tables)
report.style.format({"fail_rate": "{:.4%}"})

## 3. Kiểm tra nào chặn pipeline?

In [ ]:
# explore
blocking_checks(report)

In [ ]:
# explore: 23 đơn có timeline không thể xảy ra trông thế nào?
o = tables["orders"]
o[o.order_delivered_carrier_date > o.order_delivered_customer_date][
    ["order_id", "order_status", "order_purchase_timestamp", "order_delivered_carrier_date",
     "order_delivered_customer_date"]].head()

## 4. Đọc báo cáo

Viết 4–6 gạch đầu dòng: kiểm tra nào chặn và vì sao; kiểm tra nào bằng 0 và điều đó cho phép làm
gì; hai cảnh báo có hệ quả lớn nhất cho người phân tích sau; kiểm tra nào chỉ là thông tin.